# Notebook 3 – Categorical Feature Engineering

## Learning objectives
Identify categorical variable types and choose suitable encoding methods. Understand cardinality, rare categories, unknown categories, and leakage risks.

## 1. Categorical Variables

A categorical variable represents a group, label, or class rather than a measured numeric quantity.

### Example
Customer city, payment method, product category, and subscription plan are categorical variables.

In [1]:
import pandas as pd

customers = pd.DataFrame({
    "customer": [1, 2, 3, 4, 5, 6, 7, 8],
    "city": ["Chennai", "Salem", "Chennai", "Madurai", "Salem", "Trichy", "Chennai", "Other"],
    "membership": ["Bronze", "Silver", "Gold", "Gold", "Bronze", "Silver", "Gold", "Bronze"],
    "auto_renew": ["Yes", "No", "Yes", "Yes", "No", "No", "Yes", "No"],
    "payment": ["UPI", "Card", "Cash", "UPI", "Card", "UPI", "Wallet", "Cash"]
})
customers

,customer,city,membership,auto_renew,payment
0,1,Chennai,Bronze,Yes,UPI
1,2,Salem,Silver,No,Card
2,3,Chennai,Gold,Yes,Cash
3,4,Madurai,Gold,Yes,UPI
4,5,Salem,Bronze,No,Card
5,6,Trichy,Silver,No,UPI
6,7,Chennai,Gold,Yes,Wallet
7,8,Other,Bronze,No,Cash


### Output explanation
`city`, `membership`, `auto_renew`, and `payment` are categorical. Their values represent labels or ordered groups.

## 2. Nominal Variables

Nominal categories have no natural order. Assigning numbers to them must not imply ranking.

### Example
City and payment method are nominal variables.

## 3. Ordinal Variables

Ordinal categories have a meaningful order, but the distance between levels is not necessarily equal.

### Example
Membership levels can be ordered as Bronze < Silver < Gold.

## 4. Binary Variables

Binary variables contain two possible categories.

### Example
Auto-renew can be Yes or No.

In [2]:
print("Nominal example:", customers["city"].unique())
print("Ordinal example:", customers["membership"].unique())
print("Binary example:", customers["auto_renew"].unique())

Nominal example: <ArrowStringArray>
['Chennai', 'Salem', 'Madurai', 'Trichy', 'Other']
Length: 5, dtype: str
Ordinal example: <ArrowStringArray>
['Bronze', 'Silver', 'Gold']
Length: 3, dtype: str
Binary example: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


### Output explanation
The type of category determines the encoding choice. Do not use ordinal encoding for nominal categories unless the numeric order has real meaning.

## 5. One-Hot Encoding

One-hot encoding creates a separate 0/1 column for each category. It is suitable for nominal variables with manageable cardinality.

### Example
Convert each city into an indicator column.

In [3]:
one_hot = pd.get_dummies(
    customers["city"],
    prefix="city",
    dtype=int
)
one_hot

,city_Chennai,city_Madurai,city_Other,city_Salem,city_Trichy
0,1,0,0,0,0
1,0,0,0,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,0,0,0,1,0
5,0,0,0,0,1
6,1,0,0,0,0
7,0,0,1,0,0


### Output explanation
Each new column indicates whether a row belongs to that city. This avoids imposing a false ranking, but many unique categories can create many columns.

## 6. Ordinal Encoding

Ordinal encoding maps ordered categories to integers according to a known ranking.

### Example
Map Bronze, Silver, and Gold to increasing levels.

In [4]:
membership_order = {"Bronze": 0, "Silver": 1, "Gold": 2}
customers["membership_encoded"] = customers["membership"].map(membership_order)
customers[["membership", "membership_encoded"]]

,membership,membership_encoded
0,Bronze,0
1,Silver,1
2,Gold,2
3,Gold,2
4,Bronze,0
5,Silver,1
6,Gold,2
7,Bronze,0


### Output explanation
The assigned values preserve the defined order. Use this only when the category order is justified by the domain.

## 7. Label Encoding

Label encoding assigns an integer to each class. It is commonly used for a classification target label; applying it to nominal input features may accidentally suggest an order to some models.

### Example
Encode a target label, not an unordered input feature.

In [5]:
from sklearn.preprocessing import LabelEncoder

target = pd.Series(["Churn", "Stay", "Stay", "Churn"], name="target")
label_encoder = LabelEncoder()
encoded_target = label_encoder.fit_transform(target)

print("Classes:", label_encoder.classes_)
print("Encoded target:", encoded_target)

Classes: ['Churn' 'Stay']
Encoded target: [0 1 1 0]


### Output explanation
The encoder maps target classes to integer codes. The numeric codes are class identifiers, not a ranking of the outcomes.

## 8. Frequency Encoding

Frequency encoding replaces a category with how often it appears in the training data. It can be useful for high-cardinality categories, but equal frequencies do not mean categories are equivalent.

### Example
Encode city using its training frequency.

In [6]:
city_frequency = customers["city"].value_counts(normalize=True)
customers["city_frequency"] = customers["city"].map(city_frequency)
customers[["city", "city_frequency"]]

,city,city_frequency
0,Chennai,0.375
1,Salem,0.250
2,Chennai,0.375
3,Madurai,0.125
4,Salem,0.250
5,Trichy,0.125
6,Chennai,0.375
7,Other,0.125


### Output explanation
The value represents the proportion of rows belonging to that city in this sample. In model evaluation, calculate the mapping from training data and handle unseen categories separately.

## 9. Target Encoding

Target encoding replaces each category with a statistic of the target, often the target mean for binary or regression problems. It can be useful for high-cardinality categories but has a strong leakage risk.

### Example
Illustrate category-level churn rate using a separate training table.

In [7]:
train = pd.DataFrame({
    "city": ["Chennai", "Chennai", "Salem", "Salem", "Madurai", "Madurai"],
    "churn": [1, 0, 1, 1, 0, 0]
})

target_mean_map = train.groupby("city")["churn"].mean()
train["city_target_encoded_demo"] = train["city"].map(target_mean_map)
train

,city,churn,city_target_encoded_demo
0,Chennai,1,0.5
1,Chennai,0,0.5
2,Salem,1,1.0
3,Salem,1,1.0
4,Madurai,0,0.0
5,Madurai,0,0.0


### Output explanation
The encoded value is the observed churn rate for each city in this tiny training sample. In real work, use out-of-fold or cross-fitted target encoding and never calculate the mapping using validation/test targets.

## 10. Rare Category Grouping

Rare category grouping combines infrequent labels into a shared category such as `Other`. It can reduce sparse columns and simplify high-cardinality features.

### Example
Group cities appearing fewer than two times.

In [8]:
city_counts = customers["city"].value_counts()
rare_cities = city_counts[city_counts < 2].index

customers["city_grouped"] = customers["city"].where(
    ~customers["city"].isin(rare_cities),
    "Other"
)
customers[["city", "city_grouped"]]

,city,city_grouped
0,Chennai,Chennai
1,Salem,Salem
2,Chennai,Chennai
3,Madurai,Other
4,Salem,Salem
5,Trichy,Other
6,Chennai,Chennai
7,Other,Other


### Output explanation
Categories with low counts are grouped into `Other`. In a predictive pipeline, determine rare categories from training data only.

## 11. Category Cardinality

Cardinality is the number of distinct categories in a variable. High cardinality can produce large one-hot encoded datasets.

### Example
Count unique values in each categorical column.

In [9]:
categorical_columns = ["city", "membership", "auto_renew", "payment"]
cardinality = customers[categorical_columns].nunique().sort_values(ascending=False)
cardinality

city          5
payment       4
membership    3
auto_renew    2
dtype: int64

### Output explanation
The result shows how many distinct labels each column contains. High cardinality may call for grouping, frequency encoding, or carefully validated target encoding.

## 12. Handling Unknown Categories

An unknown category is a value that appears during validation or production but was not present during training. Encoders should have an explicit strategy for it.

### Example
Use OneHotEncoder with `handle_unknown="ignore"`.

In [10]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cities = pd.DataFrame({"city": ["Chennai", "Salem", "Madurai"]})
test_cities = pd.DataFrame({"city": ["Chennai", "Coimbatore"]})

encoder.fit(train_cities[["city"]])
encoded_test = encoder.transform(test_cities[["city"]])

print("Encoded columns:", encoder.get_feature_names_out(["city"]))
print(pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(["city"])))

Encoded columns: ['city_Chennai' 'city_Madurai' 'city_Salem']
   city_Chennai  city_Madurai  city_Salem
0           1.0           0.0         0.0
1           0.0           0.0         0.0


### Output explanation
The unseen `Coimbatore` value is represented by zeros across the learned city columns rather than causing an error. `sparse_output` is supported by recent scikit-learn versions.

## 13. Combining Categories

Combining categories creates a broader label when the grouping has a valid business meaning.

### Example
Group several payment methods into digital and non-digital.

In [11]:
payment_group = {
    "UPI": "Digital",
    "Card": "Digital",
    "Wallet": "Digital",
    "Cash": "Non-digital"
}
customers["payment_group"] = customers["payment"].map(payment_group)
customers[["payment", "payment_group"]]

,payment,payment_group
0,UPI,Digital
1,Card,Digital
2,Cash,Non-digital
3,UPI,Digital
4,Card,Digital
5,UPI,Digital
6,Wallet,Digital
7,Cash,Non-digital


### Output explanation
The grouped feature may reduce category complexity. The grouping should reflect the business context and should not hide distinctions that matter to the target.